# LeetCode 1727 — Largest Submatrix With Rearrangements bằng Python

Link đề: https://leetcode.com/problems/largest-submatrix-with-rearrangements/ · Độ khó: **Medium** · Chủ đề: Array, Greedy, Sorting, Matrix

Skeleton LeetCode:

```python
class Solution:
    def largestSubmatrix(self, matrix: List[List[int]]) -> int:
        ...
```

Lộ trình: **thử mọi hoán vị cột** (vét cạn thuần, `n ≤ 7`) → **duyệt mọi dải hàng liên tiếp** và đếm cột toàn 1 (`O(m² n)`) → **chiều cao cột + sắp xếp giảm dần mỗi hàng** (`O(m · n log n)`) → **giữ thứ tự cột từ hàng trước** để khỏi sắp lại (`O(m · n)`).

## 1. Mục tiêu học

1. Hiểu ý nghĩa "được **đổi chỗ toàn bộ cột**": các hàng giữ nguyên thứ tự, chỉ thứ tự cột thay đổi.
2. Quy bài toán về: với mỗi hàng đáy `r`, mỗi cột có **chiều cao** `h[c]` = số số 1 liên tiếp kết thúc tại hàng `r`; sắp các cột theo chiều cao giảm dần ⇒ diện tích tốt nhất `max_k k · h_sorted[k−1]`.
3. Kỹ thuật **mảng chiều cao (histogram)** cập nhật từng hàng — quen thuộc từ bài *Maximal Rectangle*.
4. Tối ưu từ `O(n log n)` mỗi hàng xuống `O(n)` bằng cách **tái sử dụng thứ tự** của hàng trước.
5. Kiểm chứng bằng vét cạn hoán vị cột và bằng cách duyệt dải hàng.

## 2. Đề bài & ví dụ

Cho ma trận nhị phân `matrix` kích thước `m × n`. Bạn được phép **sắp xếp lại các cột** theo thứ tự bất kỳ.

Trả về **diện tích** của ma trận con lớn nhất mà **mọi phần tử đều bằng 1**, sau khi sắp xếp cột một cách tối ưu.

**Ví dụ 1:** `matrix = [[0,0,1],[1,1,1],[1,0,1]]` → `4`.

**Ví dụ 2:** `matrix = [[1,0,1,0,1]]` → `3`.

**Ví dụ 3:** `matrix = [[1,1,0],[1,0,1]]` → `2`. (Phải di chuyển **cả cột**, không thể đạt diện tích lớn hơn 2.)

**Ràng buộc:**
- `m == matrix.length`, `n == matrix[i].length`
- `1 ≤ m · n ≤ 10⁵`
- `matrix[i][j]` là `0` hoặc `1`.

In [1]:
import random
import time
from itertools import permutations
from typing import List

examples = [
    ([[0, 0, 1], [1, 1, 1], [1, 0, 1]], 4),
    ([[1, 0, 1, 0, 1]], 3),
    ([[1, 1, 0], [1, 0, 1]], 2),
]
for mat, want in examples:
    print(f"matrix={mat} -> kỳ vọng {want}")

matrix=[[0, 0, 1], [1, 1, 1], [1, 0, 1]] -> kỳ vọng 4
matrix=[[1, 0, 1, 0, 1]] -> kỳ vọng 3
matrix=[[1, 1, 0], [1, 0, 1]] -> kỳ vọng 2


## 3. Kiến thức nền

### 3.1 Ma trận dạng list lồng nhau và truy cập cột

`matrix[r][c]` là hàng `r`, cột `c`. Lấy một cột: `[row[c] for row in matrix]`. Hoán vị cột theo `perm`: `[[row[c] for c in perm] for row in matrix]`.

In [2]:
mat = [[1, 1, 0], [1, 0, 1]]
perm = (2, 0, 1)
print([[row[c] for c in perm] for row in mat])

[[0, 1, 1], [1, 1, 0]]


### 3.2 Hình chữ nhật toàn 1 lớn nhất trong ma trận **cố định** (histogram)

Với hàng đáy `r`, `h[c]` = số số 1 liên tiếp tính từ hàng `r` lên trên ở cột `c`. Cập nhật: `h[c] = h[c] + 1` nếu `matrix[r][c] == 1`, ngược lại `0`. (Khi không được đổi cột, phải dùng stack đơn điệu để tìm hình chữ nhật lớn nhất trong histogram — bài 84/85.)

In [3]:
def heights_per_row(mat):
    n = len(mat[0]); h = [0] * n; out = []
    for row in mat:
        h = [h[c] + 1 if row[c] else 0 for c in range(n)]
        out.append(h)
    return out

for r, h in enumerate(heights_per_row(examples[0][0])):
    print(f"hàng đáy {r}: h = {h}")

hàng đáy 0: h = [0, 0, 1]
hàng đáy 1: h = [1, 1, 2]
hàng đáy 2: h = [2, 0, 3]


### 3.3 Sắp xếp giảm dần và "chọn k cột cao nhất"

Nếu được tùy ý xếp cột, hình chữ nhật đáy ở hàng `r` gồm `k` cột có chiều cao `≥ t` có diện tích `k · t`. Tốt nhất: sắp `h` giảm dần thành `s`, thử mọi `k`: diện tích `k · s[k−1]` (chiều cao bị giới hạn bởi cột thấp nhất trong `k` cột cao nhất).

In [4]:
h = [2, 1, 3]
s = sorted(h, reverse=True)
print("s =", s, "| diện tích với k = 1..3:", [(k, k * s[k - 1]) for k in range(1, 4)])

s = [3, 2, 1] | diện tích với k = 1..3: [(1, 3), (2, 4), (3, 3)]


### 3.4 Phép AND theo từng cột cho một dải hàng

Một cột "toàn 1 trên dải hàng `[r1, r2]`" ⇔ AND các phần tử của cột trên dải bằng 1. Cập nhật dần khi mở rộng `r2`.

In [5]:
mat = examples[0][0]
alive = [1, 1, 1]
for r2 in range(3):
    alive = [alive[c] & mat[r2][c] for c in range(3)]
    print(f"dải [0, {r2}]: cột toàn 1 = {[c for c in range(3) if alive[c]]}")

dải [0, 0]: cột toàn 1 = [2]
dải [0, 1]: cột toàn 1 = [2]
dải [0, 2]: cột toàn 1 = [2]


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 1 `[[0,0,1],[1,1,1],[1,0,1]]`. Chiều cao theo hàng đáy:

| hàng đáy | h | sắp giảm | k·s[k−1] | tốt nhất |
|---|---|---|---|---|
| 0 | [0,0,1] | [1,0,0] | 1, 0, 0 | 1 |
| 1 | [1,1,2] | [2,1,1] | 2, 2, 3 | 3 |
| 2 | [2,0,3] | [3,2,0] | 3, **4**, 0 | **4** |

Ở hàng đáy 2: cột 2 cao 3, cột 0 cao 2 ⇒ đặt hai cột này cạnh nhau ⇒ hình chữ nhật `2 × 2 = 4` (hàng 1–2, hai cột). ✓

**Vì sao chỉ cần nhìn chiều cao?** Ma trận con toàn 1 sau khi xếp cột = một **dải hàng liên tiếp** `[r1, r2]` × một **tập cột** (tùy ý, vì xếp được cạnh nhau) mà toàn 1 trên dải đó. Với hàng đáy `r2`, cột `c` toàn 1 trên `[r1, r2]` ⇔ `h[c] ≥ r2 − r1 + 1`.

## 5. Cách 1 — Brute Force / baseline: thử mọi hoán vị cột

**Ý tưởng:** với mỗi hoán vị cột, tìm hình chữ nhật toàn 1 lớn nhất trong ma trận **cố định** (vét cạn mọi hình chữ nhật bằng tổng tiền tố 2D). Lấy max. Bám sát định nghĩa nhất.

In [6]:
def max_rect_fixed(mat):
    """Hình chữ nhật toàn 1 lớn nhất trong ma trận cố định (vét cạn bằng prefix sum 2D)."""
    m, n = len(mat), len(mat[0])
    P = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m):
        for j in range(n):
            P[i + 1][j + 1] = P[i][j + 1] + P[i + 1][j] - P[i][j] + mat[i][j]
    best = 0
    for r1 in range(m):
        for r2 in range(r1, m):
            for c1 in range(n):
                for c2 in range(c1, n):
                    area = (r2 - r1 + 1) * (c2 - c1 + 1)
                    if area > best:
                        s = P[r2 + 1][c2 + 1] - P[r1][c2 + 1] - P[r2 + 1][c1] + P[r1][c1]
                        if s == area:
                            best = area
    return best

def largest_perm_brute(mat):
    n = len(mat[0])
    return max(max_rect_fixed([[row[c] for c in perm] for row in mat]) for perm in permutations(range(n)))

for mat, want in examples:
    print(largest_perm_brute(mat), "kỳ vọng", want)

4 kỳ vọng 4
3 kỳ vọng 3
2 kỳ vọng 2


**Độ phức tạp:** `n!` hoán vị × `O(m² n²)` ⇒ chỉ dùng cho `n ≤ 6–7`.

### 5.1 Cách 1b — Duyệt mọi dải hàng `[r1, r2]` (`O(m² · n)`)

Dùng nhận xét ở mục 4 (không cần hoán vị): với mỗi `r1`, mở rộng `r2` và giữ mảng `alive` (cột còn toàn 1); diện tích = `(r2 − r1 + 1) · (số cột alive)`.

In [7]:
def largest_bands(mat):
    m, n = len(mat), len(mat[0])
    best = 0
    for r1 in range(m):
        alive = [1] * n
        cnt = n
        for r2 in range(r1, m):
            row = mat[r2]
            for c in range(n):
                if alive[c] and not row[c]:
                    alive[c] = 0
                    cnt -= 1
            if cnt == 0:
                break
            best = max(best, (r2 - r1 + 1) * cnt)
    return best

for mat, want in examples:
    print(largest_bands(mat), "kỳ vọng", want)

4 kỳ vọng 4
3 kỳ vọng 3
2 kỳ vọng 2


**Độ phức tạp:** `O(m² · n)` worst. Với `m · n ≤ 10⁵`, nếu `m = 10⁵, n = 1` thì `m² = 10¹⁰` — quá chậm. **Nút thắt:** mỗi dải hàng được xét riêng; thông tin "cột còn sống bao lâu" chính là chiều cao `h[c]` — tính một lần cho mỗi hàng đáy.

In [8]:
random.seed(0)
for m, n in [(100, 100), (300, 300)]:
    mat = [[1 if random.random() < 0.9 else 0 for _ in range(n)] for _ in range(m)]
    t0 = time.perf_counter(); largest_bands(mat)
    print(f"{m}x{n}: {time.perf_counter() - t0:.3f}s")

100x100: 0.006s
300x300: 0.070s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** cố định **hàng đáy** `r`. Mọi ma trận con toàn 1 có hàng đáy `r` và chiều cao `t` dùng được đúng các cột có `h[c] ≥ t`. Vì xếp cột tùy ý, ta gom chúng lại ⇒ diện tích `t · #{c : h[c] ≥ t}`.

Sắp `h` giảm dần thành `s`. Với `k` cột cao nhất, `t` lớn nhất là `s[k−1]` ⇒ ứng viên `k · s[k−1]`. Duyệt mọi `k` cho đáp án tối ưu của hàng đáy `r`.

**Quy tắc cập nhật chiều cao:** `h[c] ← h[c] + 1` nếu `matrix[r][c] == 1`, ngược lại `h[c] ← 0`.

**Bước tối ưu thêm (Cách 3):** từ hàng `r−1` sang `r`, các cột còn 1 đều tăng chiều cao thêm 1 — **giữ nguyên thứ tự tương đối**; các cột bị 0 về chiều cao 0 — đưa xuống cuối. Cột có chiều cao 1 mới (trước đó 0, nay là 1) nằm giữa. Vậy không cần sắp lại: chỉ cần **lọc** theo thứ tự cũ.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Chiều cao + sắp xếp mỗi hàng

In [9]:
def largest_sort(mat):
    m, n = len(mat), len(mat[0])
    h = [0] * n
    best = 0
    for row in mat:
        for c in range(n):
            h[c] = h[c] + 1 if row[c] else 0
        s = sorted(h, reverse=True)
        for k in range(n):
            if s[k] == 0:
                break
            area = (k + 1) * s[k]
            if area > best:
                best = area
    return best

for mat, want in examples:
    print(largest_sort(mat), "kỳ vọng", want)

4 kỳ vọng 4
3 kỳ vọng 3
2 kỳ vọng 2


**Độ phức tạp:** Time `O(m · n log n)`, Space `O(n)`. Với `m · n ≤ 10⁵` là khoảng `1.7·10⁶` phép — rất nhanh.

### 7.2 Cách 3 — Giữ thứ tự cột từ hàng trước (`O(m · n)`)

Từng đoạn:
1. `order` = danh sách cột theo chiều cao **giảm dần** của hàng trước (ban đầu rỗng).
2. Hàng mới: `keep` = các cột trong `order` mà hàng này là 1 (chiều cao +1, thứ tự giữ nguyên); `new` = các cột hàng này là 1 nhưng trước đó chiều cao 0 (nay = 1).
3. `order = keep + new` — vẫn giảm dần vì mọi cột trong `keep` có chiều cao ≥ 2 > 1 = cột trong `new`.
4. Tính `max (k+1) · h[order[k]]`.

In [10]:
def largest_keep_order(mat):
    m, n = len(mat), len(mat[0])
    h = [0] * n
    order = []                     # các cột có h > 0, theo h giảm dần
    best = 0
    for row in mat:
        keep = []
        for c in order:
            if row[c]:
                h[c] += 1
                keep.append(c)
            else:
                h[c] = 0
        new = [c for c in range(n) if row[c] and h[c] == 0]
        for c in new:
            h[c] = 1
        order = keep + new
        for k, c in enumerate(order):
            area = (k + 1) * h[c]
            if area > best:
                best = area
    return best

for mat, want in examples:
    print(largest_keep_order(mat), "kỳ vọng", want)

4 kỳ vọng 4
3 kỳ vọng 3
2 kỳ vọng 2


**Độ phức tạp:** mỗi hàng `O(n)` ⇒ Time `O(m · n)`, Space `O(n)`.

Lưu ý dòng `new = [c for c in range(n) if row[c] and h[c] == 0]`: tại thời điểm này các cột trong `keep` đã có `h ≥ 2`, các cột bị 0 đã về `h = 0` ⇒ điều kiện `h[c] == 0` và `row[c] == 1` lọc đúng các cột "mới mọc".

### 7.3 Cách 4 — Chiều cao + đếm (counting sort) theo chiều cao

Chiều cao ≤ `m`. Với mỗi hàng, đếm `cnt[t]` = số cột có chiều cao đúng `t`, rồi duyệt `t` giảm dần, cộng dồn số cột có `h ≥ t` ⇒ diện tích `t · cộng_dồn`. Có lợi khi `m` nhỏ hơn nhiều so với `n`.

In [11]:
def largest_count(mat):
    m, n = len(mat), len(mat[0])
    h = [0] * n
    best = 0
    for row in mat:
        cnt = [0] * (m + 1)
        for c in range(n):
            h[c] = h[c] + 1 if row[c] else 0
            cnt[h[c]] += 1
        acc = 0
        for t in range(m, 0, -1):
            acc += cnt[t]
            if acc and t * acc > best:
                best = t * acc
    return best

for mat, want in examples:
    print(largest_count(mat), "kỳ vọng", want)

4 kỳ vọng 4
3 kỳ vọng 3
2 kỳ vọng 2


**Độ phức tạp:** Time `O(m · (n + m))`, Space `O(n + m)`. Tốt khi `m ≤ n`; khi `m ≫ n` thì Cách 2/3 tốt hơn.

## 8. TRACE từng bước

Trace Cách 2 trên ví dụ 1:

In [12]:
def trace_sort(mat):
    n = len(mat[0]); h = [0] * n; best = 0
    for r, row in enumerate(mat):
        h = [h[c] + 1 if row[c] else 0 for c in range(n)]
        s = sorted(h, reverse=True)
        cands = [(k + 1) * s[k] for k in range(n)]
        best = max(best, max(cands))
        print(f"hàng {r}: row={row} h={h} sắp={s} ứng viên k*s[k-1]={cands} best={best}")
    return best

trace_sort([[0, 0, 1], [1, 1, 1], [1, 0, 1]])

hàng 0: row=[0, 0, 1] h=[0, 0, 1] sắp=[1, 0, 0] ứng viên k*s[k-1]=[1, 0, 0] best=1
hàng 1: row=[1, 1, 1] h=[1, 1, 2] sắp=[2, 1, 1] ứng viên k*s[k-1]=[2, 2, 3] best=3
hàng 2: row=[1, 0, 1] h=[2, 0, 3] sắp=[3, 2, 0] ứng viên k*s[k-1]=[3, 4, 0] best=4


4

In [13]:
def trace_keep(mat):
    n = len(mat[0]); h = [0] * n; order = []; best = 0
    for r, row in enumerate(mat):
        keep = []
        for c in order:
            if row[c]: h[c] += 1; keep.append(c)
            else: h[c] = 0
        new = [c for c in range(n) if row[c] and h[c] == 0]
        for c in new: h[c] = 1
        order = keep + new
        areas = [(k + 1) * h[c] for k, c in enumerate(order)]
        best = max([best] + areas)
        print(f"hàng {r}: giữ {keep}, mới {new} -> order={order}, h theo order={[h[c] for c in order]}, "
              f"diện tích={areas}, best={best}")
    return best

trace_keep([[0, 0, 1], [1, 1, 1], [1, 0, 1]])
trace_keep([[1, 1, 0], [1, 0, 1]])

hàng 0: giữ [], mới [2] -> order=[2], h theo order=[1], diện tích=[1], best=1
hàng 1: giữ [2], mới [0, 1] -> order=[2, 0, 1], h theo order=[2, 1, 1], diện tích=[2, 2, 3], best=3
hàng 2: giữ [2, 0], mới [] -> order=[2, 0], h theo order=[3, 2], diện tích=[3, 4], best=4
hàng 0: giữ [], mới [0, 1] -> order=[0, 1], h theo order=[1, 1], diện tích=[1, 2], best=2
hàng 1: giữ [0], mới [2] -> order=[0, 2], h theo order=[2, 1], diện tích=[2, 2], best=2


2

Ví dụ 3 `[[1,1,0],[1,0,1]]`: hàng 1 có `h = [2,0,1]` ⇒ sắp `[2,1,0]` ⇒ ứng viên `2, 2, 0` ⇒ `2`. Không thể lấy cột 0 và 2 cao 2 vì cột 2 ở hàng 0 là 0.

## 9. Code nộp LeetCode

In [14]:
class Solution:
    def largestSubmatrix(self, matrix: List[List[int]]) -> int:
        n = len(matrix[0])
        h = [0] * n
        best = 0
        for row in matrix:
            for c in range(n):
                h[c] = h[c] + 1 if row[c] else 0
            for k, v in enumerate(sorted(h, reverse=True)):
                if v == 0:
                    break
                if (k + 1) * v > best:
                    best = (k + 1) * v
        return best

sol = Solution()
for mat, want in examples:
    got = sol.largestSubmatrix(mat)
    print(got, "OK" if got == want else "SAI")

4 OK
3 OK
2 OK


## 10. Vì sao thuật toán đúng

**Mô tả lời giải tối ưu.** Sau khi xếp cột, một ma trận con toàn 1 là tích của một dải hàng liên tiếp `R = [r1, r2]` và một **khối cột liên tiếp** (trong thứ tự mới). Vì thứ tự cột tùy ý, khối cột liên tiếp có thể là **bất kỳ tập cột** `C` nào. Điều kiện: mọi cột trong `C` toàn 1 trên `R`. Vậy đáp án = `max_{R, C} |R| · |C|` với `C ⊆ {c : cột c toàn 1 trên R}`.

**Quy về chiều cao.** Cố định `r2` (hàng đáy) và độ cao `t = |R|`. Cột `c` toàn 1 trên `[r2−t+1, r2]` ⇔ `h_{r2}[c] ≥ t`. Nên giá trị tốt nhất với đáy `r2`: `max_t t · #{c : h[c] ≥ t}`.

**Dùng mảng sắp xếp.** Với `s` giảm dần: `#{c : h[c] ≥ s[k−1]} ≥ k`. Mặt khác, với `t` bất kỳ, nếu `#{h ≥ t} = k` thì `s[k−1] ≥ t` ⇒ `k · s[k−1] ≥ k · t`. Vậy `max_k k · s[k−1] = max_t t · #{h ≥ t}`. ✓

**Cách 3 đúng** vì `order` luôn là các cột có `h > 0` theo `h` giảm dần (quy nạp: `keep` giữ thứ tự cũ và mọi phần tử tăng đúng 1; `new` có `h = 1` ≤ mọi phần tử của `keep` (≥ 2)).

## 11. Phân tích Time / Space

Gọi `m` số hàng, `n` số cột, `m · n ≤ 10⁵`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Hoán vị cột | `O(n! · m² n²)` | `O(mn)` | `n ≤ 6` |
| 1b. Duyệt dải hàng | `O(m² · n)` | `O(n)` | chậm khi `m` lớn |
| 2. Chiều cao + sort | `O(m · n log n)` | `O(n)` | **nộp bài** |
| 3. Giữ thứ tự | `O(m · n)` | `O(n)` | tối ưu lý thuyết |
| 4. Đếm theo chiều cao | `O(m · (n + m))` | `O(n + m)` | tốt khi `m ≤ n` |

In [15]:
random.seed(1)
for m, n in [(316, 316), (10**5, 1), (1, 10**5), (100, 1000)]:
    mat = [[1 if random.random() < 0.8 else 0 for _ in range(n)] for _ in range(m)]
    res = []
    line = f"{m:6d}x{n:<6d}:"
    for name, fn in [("sort", largest_sort), ("keep", largest_keep_order), ("count", largest_count)]:
        if name == "count" and m > 1000:
            continue
        t0 = time.perf_counter(); r = fn(mat); res.append(r)
        line += f" {name}={r} ({time.perf_counter() - t0:.3f}s)"
    assert len(set(res)) == 1
    print(line)

   316x316   : sort=630 (0.012s) keep=630 (0.010s) count=630 (0.006s)


100000x1     : sort=51 (0.019s) keep=51 (0.024s)
     1x100000: sort=79998 (0.009s) keep=79998 (0.008s) count=79998 (0.006s)


   100x1000  : sort=1768 (0.012s) keep=1768 (0.010s) count=1768 (0.005s)


## 12. Bộ test

In [16]:
fns = [largest_bands, largest_sort, largest_keep_order, largest_count, lambda a: Solution().largestSubmatrix(a)]

for mat, want in examples:
    assert largest_perm_brute(mat) == want
    for fn in fns:
        assert fn(mat) == want

edge = [([[0]], 0), ([[1]], 1), ([[1], [1], [0], [1]], 2), ([[1, 1, 1]], 3), ([[0, 0], [0, 0]], 0),
        ([[1, 0], [0, 1]], 1), ([[1, 1], [1, 1]], 4)]
for mat, want in edge:
    assert largest_perm_brute(mat) == want, mat
    for fn in fns:
        assert fn(mat) == want, (fn, mat)

random.seed(2024)
for _ in range(400):
    m, n = random.randint(1, 5), random.randint(1, 5)
    p = random.choice([0.3, 0.6, 0.85])
    mat = [[1 if random.random() < p else 0 for _ in range(n)] for _ in range(m)]
    want = largest_perm_brute(mat)
    for fn in fns:
        assert fn(mat) == want, (fn, mat)
print("ví dụ + edge + 400 ma trận ngẫu nhiên khớp vét cạn hoán vị cột")

ví dụ + edge + 400 ma trận ngẫu nhiên khớp vét cạn hoán vị cột


In [17]:
random.seed(7)
for _ in range(100):
    m, n = random.randint(1, 40), random.randint(1, 40)
    p = random.random()
    mat = [[1 if random.random() < p else 0 for _ in range(n)] for _ in range(m)]
    r = largest_bands(mat)
    assert r == largest_sort(mat) == largest_keep_order(mat) == largest_count(mat)
print("ma trận tới 40x40: các cách khớp duyệt dải hàng")

ma trận tới 40x40: các cách khớp duyệt dải hàng


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Sắp xếp **từng hàng** của ma trận gốc (đổi chỗ phần tử trong hàng độc lập)

Đề chỉ cho đổi **cả cột**. Sắp từng hàng riêng lẻ tương đương đổi chỗ phần tử tự do — cho đáp án quá lớn.

In [18]:
def wrong_sort_rows(mat):
    sorted_mat = [sorted(row, reverse=True) for row in mat]   # SAI: mỗi hàng xếp riêng
    return max_rect_fixed(sorted_mat)

mat = [[1, 1, 0], [1, 0, 1]]
print("sai =", wrong_sort_rows(mat), "| đúng =", Solution().largestSubmatrix(mat))

sai = 4 | đúng = 2


**Sửa:** sắp **chiều cao** `h` của từng hàng đáy (mỗi `h[c]` gắn với cả một cột phía trên), không sắp giá trị gốc.

### Lỗi 2 — Sắp xếp `h` **tại chỗ** rồi dùng tiếp cho hàng sau

`h.sort()` làm mất liên kết "chiều cao ↔ cột"; ở hàng sau, `h[c]` không còn là chiều cao của cột `c`.

In [19]:
def wrong_inplace_sort(mat):
    n = len(mat[0]); h = [0] * n; best = 0
    for row in mat:
        for c in range(n):
            h[c] = h[c] + 1 if row[c] else 0
        h.sort(reverse=True)                     # SAI: phá thứ tự cột
        for k in range(n):
            best = max(best, (k + 1) * h[k])
    return best

random.seed(13)
for _ in range(5000):
    m, n = random.randint(1, 4), random.randint(2, 4)
    mat = [[random.randint(0, 1) for _ in range(n)] for _ in range(m)]
    if wrong_inplace_sort(mat) != Solution().largestSubmatrix(mat):
        print("phản ví dụ:", mat, "| sai =", wrong_inplace_sort(mat), "| đúng =", Solution().largestSubmatrix(mat))
        break

phản ví dụ: [[0, 1, 1], [1, 1, 1], [0, 1, 1]] | sai = 4 | đúng = 6


**Sửa:** `s = sorted(h, reverse=True)` (bản sao).

### Lỗi 3 — Không reset chiều cao về 0 khi gặp số 0

Viết `h[c] += row[c]` ⇒ chiều cao đếm **tổng** số 1 trong cột chứ không phải số 1 **liên tiếp**.

In [20]:
def wrong_no_reset(mat):
    n = len(mat[0]); h = [0] * n; best = 0
    for row in mat:
        for c in range(n):
            h[c] += row[c]                       # SAI: phải reset khi row[c] == 0
        s = sorted(h, reverse=True)
        for k in range(n):
            best = max(best, (k + 1) * s[k])
    return best

mat = [[1, 0], [0, 1], [1, 1]]
print("sai =", wrong_no_reset(mat), "| đúng =", Solution().largestSubmatrix(mat))

sai = 4 | đúng = 2


**Sửa:** `h[c] = h[c] + 1 if row[c] else 0`.

### Lỗi 4 — Chỉ thử `k = số cột có h > 0` (lấy tất cả cột "sống")

Lấy hết các cột có `h > 0` với chiều cao `min` của chúng — bỏ qua lựa chọn ít cột hơn nhưng cao hơn.

In [21]:
def wrong_all_alive(mat):
    n = len(mat[0]); h = [0] * n; best = 0
    for row in mat:
        for c in range(n):
            h[c] = h[c] + 1 if row[c] else 0
        alive = [x for x in h if x > 0]
        if alive:
            best = max(best, len(alive) * min(alive))   # SAI: chỉ một ứng viên
    return best

random.seed(13)
for _ in range(5000):
    m, n = random.randint(1, 4), random.randint(2, 4)
    mat = [[random.randint(0, 1) for _ in range(n)] for _ in range(m)]
    if wrong_all_alive(mat) != Solution().largestSubmatrix(mat):
        print("phản ví dụ:", mat, "| sai =", wrong_all_alive(mat), "| đúng =", Solution().largestSubmatrix(mat))
        break

phản ví dụ: [[1, 1, 0, 1], [1, 0, 1, 1]] | sai = 3 | đúng = 4


**Sửa:** thử mọi `k` với mảng sắp giảm dần.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Ma trận con toàn 1" ⇒ nghĩ ngay **mảng chiều cao theo hàng đáy** (histogram).
- Không đổi cột ⇒ stack đơn điệu (bài 84/85). **Được đổi cột** ⇒ chỉ cần **sắp chiều cao** (dễ hơn!).
- "Được hoán vị tùy ý" thường biến bài toán **vị trí** thành bài toán **đếm/sắp xếp**.
- `m · n ≤ 10⁵` (không phải `m, n ≤ 10⁵`) ⇒ cẩn thận trường hợp ma trận rất dẹt hoặc rất cao.

## 15. Biến thể của bài

1. **Không được đổi cột (LeetCode 85 Maximal Rectangle):** chiều cao + stack đơn điệu `O(mn)`.
2. **Được đổi hàng thay vì cột:** chuyển vị ma trận rồi áp dụng cùng thuật toán.
3. **Được đổi cả hàng lẫn cột:** khó hơn nhiều (bài toán biclique lớn nhất — NP-khó nói chung).
4. **Hình vuông toàn 1 lớn nhất sau khi đổi cột:** với `s` giảm dần, cạnh tốt nhất `max_k min(k, s[k−1])`.

In [22]:
def maximal_rectangle_fixed(mat):
    """Biến thể 1 (LeetCode 85): không đổi cột, dùng stack đơn điệu."""
    n = len(mat[0]); h = [0] * n; best = 0
    for row in mat:
        for c in range(n):
            h[c] = h[c] + 1 if row[c] else 0
        st = []
        for i in range(n + 1):
            cur = h[i] if i < n else 0
            while st and h[st[-1]] >= cur:
                height = h[st.pop()]
                left = st[-1] + 1 if st else 0
                best = max(best, height * (i - left))
            st.append(i)
    return best

def largest_square_rearrange(mat):
    """Biến thể 4."""
    n = len(mat[0]); h = [0] * n; best = 0
    for row in mat:
        for c in range(n):
            h[c] = h[c] + 1 if row[c] else 0
        s = sorted(h, reverse=True)
        for k in range(n):
            best = max(best, min(k + 1, s[k]))
    return best * best

random.seed(3)
for _ in range(200):
    m, n = random.randint(1, 6), random.randint(1, 6)
    mat = [[random.randint(0, 1) for _ in range(n)] for _ in range(m)]
    assert maximal_rectangle_fixed(mat) == max_rect_fixed(mat)
mat = examples[0][0]
print("không đổi cột:", maximal_rectangle_fixed(mat), "| đổi cột:", Solution().largestSubmatrix(mat),
      "| hình vuông sau đổi cột:", largest_square_rearrange(mat))

không đổi cột: 3 | đổi cột: 4 | hình vuông sau đổi cột: 4


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Hoán vị cột | định nghĩa | `O(n!·m²n²)` | `O(mn)` | kiểm chứng tí hon |
| 1b. Dải hàng | AND theo cột | `O(m²n)` | `O(n)` | kiểm chứng vừa |
| 2. Chiều cao + sort | `max k·s[k−1]` | `O(mn log n)` | `O(n)` | **nộp bài** |
| 3. Giữ thứ tự | lọc theo thứ tự cũ | `O(mn)` | `O(n)` | tối ưu |
| 4. Đếm chiều cao | counting sort | `O(m(n+m))` | `O(n+m)` | `m` nhỏ |

## 17. Bài tập (từ dễ đến khó)

**Bài 1 (đoán output):** `largestSubmatrix([[1,1,1],[1,1,0],[1,0,0]])`?

**Bài 2 (tính tay):** với `[[1,0,1,1],[1,1,0,1],[1,1,1,1]]` lập bảng `h` và `s` theo từng hàng đáy.

**Bài 3 (điền TODO):** hoàn thiện `best_for_heights(h)` trả về `max_k k · s[k−1]` cho một mảng chiều cao.

**Bài 4 (tự viết lại):** viết `largest_rows_rearranged(matrix)` cho biến thể **được đổi hàng** (không đổi cột).

**Bài 5 (mini challenge):** trả về không chỉ diện tích mà cả **một cách xếp cột** (hoán vị) và tọa độ `(r1, r2, c1, c2)` của ma trận con tối ưu trong ma trận đã xếp; kiểm tra bằng `max_rect_fixed`.

In [23]:
def best_for_heights(h):
    # TODO: sắp giảm dần, trả về max (k+1) * s[k]
    pass

## 18. Gợi ý 3 mức

**Bài 1:** (1) Hàng đáy 0: `h = [1,1,1]` ⇒ 3. (2) Hàng đáy 1: `[2,2,0]` ⇒ 4. (3) Hàng đáy 2: `[3,0,0]` ⇒ 3; đáp án `4`.

**Bài 2:** (1) `h0 = [1,0,1,1]`, `h1 = [2,1,0,2]`, `h2 = [3,2,1,3]`. (2) `s2 = [3,3,2,1]` ⇒ ứng viên `3, 6, 6, 4`. (3) Đáp án `6`.

**Bài 3:** (1) `s = sorted(h, reverse=True)`. (2) `max((k + 1) * s[k] for k in range(len(s)))`. (3) Mảng rỗng/toàn 0 ⇒ 0.

**Bài 4:** (1) Đổi hàng ⇔ đổi cột của ma trận chuyển vị. (2) `list(zip(*matrix))`. (3) Gọi lại lời giải.

**Bài 5:** (1) Ghi lại `(r, k)` tốt nhất. (2) Hoán vị: `k` cột cao nhất ở hàng `r` đặt trước, các cột còn lại sau. (3) Ma trận con: hàng `r − s[k−1] + 1 .. r`, cột `0 .. k−1`.

## 19. Đáp án tham khảo

In [24]:
# Bài 1
assert Solution().largestSubmatrix([[1, 1, 1], [1, 1, 0], [1, 0, 0]]) == 4

# Bài 2
assert Solution().largestSubmatrix([[1, 0, 1, 1], [1, 1, 0, 1], [1, 1, 1, 1]]) == 6

# Bài 3
def best_for_heights_ans(h):
    s = sorted(h, reverse=True)
    return max([(k + 1) * s[k] for k in range(len(s))] + [0])
assert best_for_heights_ans([2, 0, 3]) == 4 and best_for_heights_ans([]) == 0

# Bài 4
def largest_rows_rearranged(matrix):
    t = [list(col) for col in zip(*matrix)]
    return Solution().largestSubmatrix(t)

def brute_rows(mat):
    m = len(mat)
    return max(max_rect_fixed([mat[r] for r in perm]) for perm in permutations(range(m)))

random.seed(91)
for _ in range(200):
    m, n = random.randint(1, 5), random.randint(1, 5)
    mat = [[random.randint(0, 1) for _ in range(n)] for _ in range(m)]
    assert largest_rows_rearranged(mat) == brute_rows(mat)
print("Bài 1-4: OK")

Bài 1-4: OK


In [25]:
# Bài 5 — trả về cách xếp cột và ma trận con
def largest_with_witness(mat):
    m, n = len(mat), len(mat[0]); h = [0] * n
    best, info = 0, None
    for r, row in enumerate(mat):
        for c in range(n):
            h[c] = h[c] + 1 if row[c] else 0
        order = sorted(range(n), key=lambda c: -h[c])
        for k in range(n):
            area = (k + 1) * h[order[k]]
            if area > best:
                best, info = area, (r, k + 1, h[order[k]], order[:])
    if not info:
        return 0, list(range(n)), None
    r, k, t, order = info
    return best, order, (r - t + 1, r, 0, k - 1)

random.seed(92)
for _ in range(300):
    m, n = random.randint(1, 6), random.randint(1, 6)
    mat = [[1 if random.random() < 0.7 else 0 for _ in range(n)] for _ in range(m)]
    area, perm, rect = largest_with_witness(mat)
    re = [[row[c] for c in perm] for row in mat]
    assert area == Solution().largestSubmatrix(mat) == max_rect_fixed(re)
    if rect:
        r1, r2, c1, c2 = rect
        assert all(re[i][j] == 1 for i in range(r1, r2 + 1) for j in range(c1, c2 + 1))
        assert (r2 - r1 + 1) * (c2 - c1 + 1) == area
print("Bài 5: OK; ví dụ 1 ->", largest_with_witness(examples[0][0]))

Bài 5: OK; ví dụ 1 -> (4, [2, 0, 1], (1, 2, 0, 1))


## 20. Template nhớ nhanh + Checklist

```python
# Ma trận con toàn 1 lớn nhất khi được đổi chỗ cột
h = [0] * n; best = 0
for row in matrix:
    for c in range(n):
        h[c] = h[c] + 1 if row[c] else 0          # chiều cao 1 liên tiếp
    for k, v in enumerate(sorted(h, reverse=True)):  # bản sao, không sort tại chỗ
        if v == 0: break
        best = max(best, (k + 1) * v)              # k+1 cột cao nhất, cao v
return best
```

**Checklist trước khi submit:**
- [ ] Chiều cao reset về 0 khi gặp số 0.
- [ ] Sắp **bản sao** của `h` (không phá liên kết cột).
- [ ] Thử mọi `k`, không chỉ lấy hết các cột còn sống.
- [ ] Không sắp từng hàng của ma trận gốc.
- [ ] Thử ma trận 1 hàng, 1 cột, toàn 0.